<a href="https://colab.research.google.com/github/francescods04/f-llm-fpga/blob/main/notebooks/colab_35b_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# F-LLM FPGA — Qwen3.6-35B-A3B Real Model Baseline on Colab

**Requirements:** Colab Pro/Pro+ with A100 (40GB or 80GB) or RTX PRO 6000.

**IMPORTANT: This model is gated/private on HuggingFace.**
Before running, you MUST add your HF token to Colab secrets:
1. Get token: https://huggingface.co/settings/tokens (scope: read)
2. In Colab, click 🔑 Secrets (left panel)
3. Add secret: Name = `HF_TOKEN`, Value = your token
4. Toggle **Notebook access** ON
5. Runtime → Restart session

This notebook measures the real target model for **Gate G1** of `docs/H100_BEAT_PLAN.md`.

## Known Issue: llama.cpp / Unsloth

Qwen3.6-35B-A3B is a **MoE model** with a custom architecture that **llama.cpp standard does NOT support** well.
- `llama.cpp main branch`: hangs indefinitely during model load (timeout after 5+ min)
- `llama.cpp mtp-clean branch`: also hangs
- Unsloth claims ~220 tok/s but their build requires **custom patches** not in public repos

**Therefore we use vLLM as the primary backend** — it has native MoE support.

**Backends available:**
1. **vLLM** (RECOMMENDED — best MoE support, auto-quant, ~30-80 tok/s on A100)
2. **transformers from git** (fallback — ~9 tok/s, slow but compatible)
3. **llama.cpp + Unsloth** (experimental — may hang, only works with Unsloth's private build)

**What you get:**
1. Real tok/s (greedy, batch=1, 64-128 new tokens)
2. Peak VRAM usage
3. Generated text sample
4. JSON export for download

**Note:** First run downloads ~70 GB of weights (FP16). This takes 10–20 minutes.


In [ ]:
# RECOMMENDED: vLLM backend (best MoE support)
import random, time
cache_bust = int(time.time())
!wget -q "https://raw.githubusercontent.com/francescods04/f-llm-fpga/main/scripts/colab_35b_vllm.py?nocache={cache_bust}" -O /tmp/colab_35b_vllm.py
%run /tmp/colab_35b_vllm.py

## Fallback 1: transformers from git

If vLLM fails, try the original transformers backend (slower, ~9 tok/s):

In [ ]:
# Fallback 1: transformers from git
import random, time
cache_bust = int(time.time())
!wget -q "https://raw.githubusercontent.com/francescods04/f-llm-fpga/main/scripts/colab_35b_baseline.py?nocache={cache_bust}" -O /tmp/colab_35b_baseline.py
%run /tmp/colab_35b_baseline.py

## Fallback 2: llama.cpp (EXPERIMENTAL — likely to hang)

If you want to try llama.cpp anyway (requires Unsloth's private build):

In [ ]:
# Fallback 2: llama.cpp (WARNING: may hang with Qwen3.6 MoE)
import random, time
cache_bust = int(time.time())
!wget -q "https://raw.githubusercontent.com/francescods04/f-llm-fpga/main/scripts/colab_35b_ultra.py?nocache={cache_bust}" -O /tmp/colab_35b_ultra.py
%run /tmp/colab_35b_ultra.py

## Download Result

After the run completes, find `35b_baseline_*.json` in the Files panel on the left and download it.

Upload it back to the local project under `benchmarks/` to lock the Gate G1 measurement.